In [ ]:
# Setup and data
import pandas as pd, numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

P = "/kaggle/input/competitions/network-intrusion-detection2-ml-challenge/"
tr, te = pd.read_csv(P + "train.csv"), pd.read_csv(P + "test.csv")
y = tr.pop("Target").values
ids = te["id"]
tr = tr.drop(columns=["id"], errors="ignore")
te = te.drop(columns=["id"], errors="ignore")
print(tr.shape, te.shape)
print(pd.Series(y).value_counts(normalize=True).round(3))

In [ ]:
# Preprocessing and feature engineering
# shared category codes so train and test encodings match
for c in ["proto", "service"]:
    cats = pd.concat([tr[c], te[c]]).astype("category").cat.categories
    tr[c] = pd.Categorical(tr[c], cats).codes
    te[c] = pd.Categorical(te[c], cats).codes

def fe(d):
    d = d.copy()
    d["tot_pkts"]  = d["fwd_pkts_tot"] + d["bwd_pkts_tot"]
    d["pkt_ratio"] = d["fwd_pkts_tot"] / (d["bwd_pkts_tot"] + 1)
    d["syn_frac"]  = d["flow_SYN_flag_count"] / (d["tot_pkts"] + 1)
    return d.replace([np.inf, -np.inf], np.nan).fillna(0)

tr, te = fe(tr), fe(te)
classes = np.unique(y)

In [ ]:
# Validation: pseudo-profiles + GroupKFold
num = tr.select_dtypes("number").columns
Z = StandardScaler().fit_transform(np.log1p(tr[num].clip(lower=0)))
groups = np.zeros(len(tr), int)
for i, cls in enumerate(classes):
    m = y == cls
    groups[m] = KMeans(min(5, m.sum()), n_init=3, random_state=0).fit_predict(Z[m]) + 10 * i

In [ ]:
# Model comparison under group-CV
def lgb_basic():
    return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15,
        min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
        class_weight="balanced", verbose=-1, n_jobs=-1)

def rf():
    return RandomForestClassifier(300, min_samples_leaf=5, class_weight="balanced",
                                  n_jobs=-1, random_state=0)

configs = {
    "lgb_keep_orig_p": (lgb_basic, []),
    "lgb_drop_orig_p": (lgb_basic, ["id.orig_p"]),
    "rf_drop_orig_p":  (rf,        ["id.orig_p"]),
}
for name, (mk, drop) in configs.items():
    X = tr.drop(columns=drop)
    oof = np.empty(len(tr), object)
    for a, b in GroupKFold(5).split(X, y, groups):
        oof[b] = mk().fit(X.iloc[a], y[a]).predict(X.iloc[b])
    print(f"{name:18s} group-CV macro F1 = {f1_score(y, oof, average='macro'):.4f}")

In [ ]:
# Final model: regularized, bagged LightGBM
Xb, Xtb = tr.drop(columns=["id.orig_p"]), te.drop(columns=["id.orig_p"])

def make_bag(seed, mcs=100, col=0.5):
    return lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15,
        min_child_samples=mcs, subsample=0.7, subsample_freq=1, colsample_bytree=col,
        reg_lambda=5, extra_trees=True, class_weight="balanced",
        random_state=seed, verbose=-1, n_jobs=-1)

pt_bag = np.zeros((len(te), len(classes)))
for s in range(5):
    pt_bag += make_bag(s).fit(Xb, y).predict_proba(Xtb) / 5

pred_bag = classes[pt_bag.argmax(1)]
print(pd.Series(pred_bag).value_counts(normalize=True).round(3))
pd.DataFrame({"id": ids, "Target": pred_bag}).to_csv("submission_bag.csv", index=False)

In [ ]:
# Hedge: pseudo-labeling
conf = pt_bag.max(1) > 0.9
print("confident test rows:", round(conf.mean(), 3))
Xpl = pd.concat([Xb, Xtb[conf]], ignore_index=True)
ypl = np.r_[y, pred_bag[conf]]

pt_pl = np.zeros((len(te), len(classes)))
for s in range(5):
    pt_pl += make_bag(s).fit(Xpl, ypl).predict_proba(Xtb) / 5

pred_pl = classes[pt_pl.argmax(1)]
print(pd.Series(pred_pl).value_counts(normalize=True).round(3))
pd.DataFrame({"id": ids, "Target": pred_pl}).to_csv("submission_pl.csv", index=False)

In [ ]:
# Key finding: the training labels are confounded with "a reply exists"
ZERO_KW = ("iat", "active", "idle", "per_sec", "per_second", "duration", "down_up")

def mask_reply(d):
    d = d.copy()
    for c in d.columns:
        if c.startswith("bwd") or any(k in c for k in ZERO_KW):
            d[c] = 0
    for c in ["flow_RST_flag_count", "flow_ACK_flag_count", "flow_FIN_flag_count",
              "flow_ECE_flag_count", "flow_CWR_flag_count"]:
        d[c] = 0
    for s in ["min", "max", "tot", "avg"]:
        d["flow_pkts_payload." + s] = d["fwd_pkts_payload." + s]
    d["flow_pkts_payload.std"] = 0
    d["tot_pkts"] = d["fwd_pkts_tot"]; d["pkt_ratio"] = d["fwd_pkts_tot"]
    d["syn_frac"] = d["flow_SYN_flag_count"] / (d["tot_pkts"] + 1)
    return d

# sanity check 1: how many test rows carry the signature, and what is it in train?
sig = lambda d: (d["fwd_pkts_payload.tot"] == 120) & (d["fwd_init_window_size"] == 64)
print("test rows with signature:", int(sig(te).sum()), "| share of test:", round(sig(te).mean(), 3))
print("train classes with signature:", pd.Series(y[sig(tr).values]).value_counts().to_dict())
print("test signature rows with no reply:", int((sig(te) & (te.bwd_pkts_tot == 0)).sum()))

In [ ]:
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
def build_aug(X, yy, only_avail=True, frac=0.2):
    q = ((X.fwd_pkts_tot == 1) & (X.bwd_pkts_tot >= 1)).values
    if only_avail:
        q = q & (yy == "Availability")
    keep = q & (rng.random(len(yy)) < frac)
    return pd.concat([X, mask_reply(X[keep])], ignore_index=True), np.r_[yy, yy[keep]]

# sanity check 2: simulate the test condition on held-out train rows (mask their replies)
idx = np.arange(len(Xb))
a, b = train_test_split(idx, test_size=0.2, random_state=0, stratify=y)
Xv, yv = Xb.iloc[b], y[b]
sel = ((Xv.fwd_pkts_tot == 1) & (Xv.bwd_pkts_tot >= 1)).values
Xv_m, yv_m = mask_reply(Xv[sel]), yv[sel]
for name, (Xtr_, ytr_) in {"no augmentation": (Xb.iloc[a], y[a]),
                           "with augmentation": build_aug(Xb.iloc[a], y[a])}.items():
    p = make_bag(0).fit(Xtr_, ytr_).predict(Xv_m)
    print(f"{name:18s} | Availability recall on reply-masked held-out rows: "
          f"{(p[yv_m == 'Availability'] == 'Availability').mean():.3f}")

In [ ]:
Xaug, yaug = build_aug(Xb, y)
pt_aug = np.zeros((len(te), len(classes)))
for s in range(5):
    pt_aug += make_bag(s).fit(Xaug, yaug).predict_proba(Xtb) / 5
pred_aug = classes[pt_aug.argmax(1)]
print(pd.Series(pred_aug).value_counts(normalize=True).round(3).to_dict())
print("agreement with plain bag:", round((pred_aug == pred_bag).mean(), 3))
pd.DataFrame({"id": ids, "Target": pred_aug}).to_csv("submission_augA.csv", index=False)